# Seção 7 (tipos `Annotated`) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`06_projeto_solucao_instrutor`](06_projeto_solucao_instrutor.ipynb) | `id_` voltou a ser `UUID4 \| None`: regressão do requisito da seção 6 |
| [`07_projeto_solucao_comentada`](07_projeto_solucao_comentada.ipynb) | tipos testados isoladamente com `TypeAdapter` |

## Novas

**1. `default` dentro do `Annotated` torna opcional todo campo que usa o tipo.**

In [1]:
from typing import Annotated, TypeVar
from pydantic import BaseModel, BeforeValidator, Field, StringConstraints, TypeAdapter, ValidationError

Quantity = Annotated[int, Field(default=0, ge=0)]


class Stock(BaseModel):
    on_hand: Quantity
    reserved: Quantity


print(f"Stock(): {Stock()!r}")
print(f"campos obrigatórios: {[name for name, info in Stock.model_fields.items() if info.is_required()]}")

Stock(): Stock(on_hand=0, reserved=0)
campos obrigatórios: []


Quem lê `on_hand: Quantity` acha que o campo é obrigatório; ele não é. Um payload sem `on_hand` vira estoque zero sem erro. Tipos
reutilizáveis carregam **restrições**; o default é decisão de cada campo (`on_hand: Quantity = 0` quando for o caso).

**2. Tipo genérico usado sem parâmetro aceita qualquer coisa nos itens.**

In [2]:
T = TypeVar("T")
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]


class Basket(BaseModel):
    items: BoundedList                 # esqueceu o [str]


print(f"Basket(items=[1, 'a', None]): {Basket(items=[1, 'a', None])!r}")

Basket(items=[1, 'a', None]): Basket(items=[1, 'a', None])


O tamanho continua validado, os itens não (`T` vira `Any`). Como o erro é silencioso, vale um teste que confere o schema dos
campos (`"items": {"type": "array", "items": {...}}` sem `items` = genérico esquecido).

**3. `StringConstraints(to_lower=True, pattern=...)`: o padrão é testado antes da conversão.**

In [3]:
Slug = Annotated[str, StringConstraints(to_lower=True, pattern=r"^[a-z0-9-]+$")]
LowerFirst = Annotated[str, StringConstraints(pattern=r"^[a-z0-9-]+$"),
                       BeforeValidator(lambda v: v.lower() if isinstance(v, str) else v)]

for label, slug_type in [("StringConstraints", Slug), ("BeforeValidator(lower)", LowerFirst)]:
    try:
        print(f"{label}: {TypeAdapter(slug_type).validate_python('Meu-Slug')!r}")
    except ValidationError as ex:
        print(f"{label}: {ex.errors()[0]['type']}")

StringConstraints: string_pattern_mismatch
BeforeValidator(lower): 'meu-slug'


Lendo a declaração, parece "converte para minúsculas e valida". Na prática o `pattern` vê `"Meu-Slug"` e recusa. Normalização que
o padrão depende dela entra como `BeforeValidator`.

## Veja também

- [`dataclass_factory/05_modelos_a_partir_de_especificacao`](../../../dataclass_factory/05_modelos_a_partir_de_especificacao.ipynb): ordem de execução
  dos metadados no `Annotated` (restrições antes de *after*; vários *before* em ordem inversa).